In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

In [2]:
event_logs = pd.read_csv('../filtered_datasets/event_logs(1).csv')
marketing_summary = pd.read_csv('../filtered_datasets/marketing_summary(1).csv')
trend_report = pd.read_csv('../filtered_datasets/trend_report(1).csv')


# 1. Customer Types

In [4]:
event_logs.head()


,user_id,event_type,event_time,product_id,amount
0,U0099,checkout,2023-06-03 04:13:00,P010,NaN
1,U0240,wishlist_add,2023-06-03 05:08:00,P020,0.0
2,U0374,profile_update,2023-06-05 06:22:00,P028,0.0
3,U0122,page_view,2023-06-06 03:45:00,P001,0.0
4,U0211,wishlist_add,2023-06-03 12:38:00,P015,0.0


In [6]:
purchase_counts = (
    event_logs[event_logs['event_type'] == 'checkout']
    .groupby('user_id')
    .size()
    .reset_index(name='purchase_count')
)
user_segments = (
    event_logs[['user_id']]
    .drop_duplicates()
    .merge(purchase_counts, on='user_id', how='left')
)
#replace missing purchases with 0
user_segments['purchase_count'] = (
    user_segments['purchase_count']
    .fillna(0)
    .astype(int)
)

#create segments
user_segments['segment'] = user_segments['purchase_count'].apply(
    lambda x:
        'Non Buyer' if x == 0 else
        'Single Buyer' if x == 1 else
        'Repeat Buyer'
)
user_segments.to_csv('user_segments.csv', index = False)

# 2. Event Logs Forecast

In [7]:
# ============================================================
# Per-product DAILY sales forecast (checkout amount only)
# Replaces the old hourly total_events/unique_users per-product
# block. Grain is daily because this answers a sales-forecasting
# question (next day / next week), matching the daily/weekly
# grain already used by marketing_summary and trend_report.
# ============================================================

# Filter to checkout events only — these are the only rows where
# 'amount' represents real revenue (per your cleaning step).
checkout_events = event_logs[event_logs['event_type'] == 'checkout'].copy()

checkout_events['date'] = pd.to_datetime(checkout_events['event_time']).dt.date
checkout_events['date'] = pd.to_datetime(checkout_events['date'])  # back to datetime for asfreq

# Aggregate sales by day and product
daily_sales = (
    checkout_events
    .groupby(['date', 'product_id'])['amount']
    .sum()
    .reset_index(name='sales')
)

forecasts_list = []

for product_id, group in daily_sales.groupby('product_id'):
    daily = group.set_index('date').sort_index().asfreq('D').fillna(0)

    p_value = adfuller(daily['sales'].dropna())[1]

    if p_value >= 0.05:
        model = SARIMAX(daily['sales'], order=(1,1,1), seasonal_order=(1,0,1,7))
        print(f"{product_id} → SARIMA")
    else:
        model = ARIMA(daily['sales'], order=(1,0,1))
        print(f"{product_id} → ARIMA")

    result = model.fit()
    forecast_values = result.forecast(steps=7).round().clip(lower=0)

    product_forecast = pd.DataFrame({
        'date': forecast_values.index,
        'sales': forecast_values.values,
        'product_id': product_id
    })
    forecasts_list.append(product_forecast)

forecast_df = pd.concat(forecasts_list, ignore_index=True)
forecast_df['type'] = 'forecast'

actual_df = daily_sales.copy()
actual_df['type'] = 'actual'

combined_sales = pd.concat([actual_df, forecast_df], ignore_index=True)

# add week column for Tableau's forecast horizon filter (day vs week)
combined_sales['week'] = pd.to_datetime(combined_sales['date']).dt.strftime('%Y-W%V')

combined_sales.to_csv('product_sales_forecast.csv', index=False)
print(combined_sales.head())

P001 → SARIMA
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.92816D+00    |proj g|=  6.12419D-02

At iterate    5    f=  6.85288D+00    |proj g|=  8.64824D-04

At iterate   10    f=  6.85287D+00    |proj g|=  8.37926D-04

At iterate   15    f=  6.85236D+00    |proj g|=  1.37677D-02

At iterate   20    f=  6.81141D+00    |proj g|=  7.23634D-02

At iterate   25    f=  6.78632D+00    |proj g|=  1.98711D-03

At iterate   30    f=  6.78525D+00    |proj g|=  3.69376D-04

At iterate   35    f=  6.78511D+00    |proj g|=  2.92111D-04

At iterate   40    f=  6.78508D+00    |proj g|=  3.88647D-04

At iterate   45    f=  6.78506D+00    |proj g|=  1.24349D-03

At iterate   50    f=  6.78502D+00    |proj g|=  1.78409D-03

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explo

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
 This problem is unconstrained.
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All 


At iterate    5    f=  1.94928D+11    |proj g|=  1.37550D+14

At iterate   10    f=  2.65941D+10    |proj g|=  6.91303D+12

At iterate   15    f=  3.62116D+09    |proj g|=  3.46881D+11

At iterate   20    f=  4.92366D+08    |proj g|=  1.73934D+10

At iterate   25    f=  6.65760D+07    |proj g|=  8.68542D+08

At iterate   30    f=  8.67395D+06    |proj g|=  4.21330D+07

At iterate   35    f=  1.01056D+06    |proj g|=  1.91673D+06

At iterate   40    f=  2.09091D+04    |proj g|=  2.56813D+04

At iterate   45    f=  1.70887D+03    |proj g|=  1.17214D+03

At iterate   50    f=  1.37453D+02    |proj g|=  4.57888D+01

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tn

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few obser

P005 → ARIMA
P006 → ARIMA
P007 → SARIMA
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  7.37834D+00    |proj g|=  6.95426D-02

At iterate    5    f=  7.30572D+00    |proj g|=  5.13410D-03

At iterate   10    f=  7.30500D+00    |proj g|=  9.08440D-05

At iterate   15    f=  7.30439D+00    |proj g|=  1.09546D-02

At iterate   20    f=  7.27957D+00    |proj g|=  5.83570D-03


...

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
 This problem is unconstrained.
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.


At iterate    5    f=  6.37995D+10    |proj g|=  1.50361D+13

At iterate   10    f=  8.70498D+09    |proj g|=  7.58744D+11

At iterate   15    f=  1.18307D+09    |proj g|=  3.80547D+10

At iterate   20    f=  1.59927D+08    |proj g|=  1.90029D+09

At iterate   25    f=  2.09411D+07    |proj g|=  9.26484D+07

At iterate   30    f=  2.50098D+06    |proj g|=  4.25815D+06

At iterate   35    f=  3.64032D+04    |proj g|=  4.22960D+04

At iterate   40    f=  3.27302D+03    |proj g|=  2.05888D+03

At iterate   45    f=  2.62393D+02    |proj g|=  8.07320D+01

At iterate   50    f=  2.44370D+01    |proj g|=  5.60155D+00

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tn

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'

P019 → SARIMA
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.10715D+16    |proj g|=  6.30838D+20

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
    5      2     18      1     0     0   1.113D+12   6.851D+09
  F =   6851013903.2453537     

CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH             


ValueError: sample size is too short to use selected regression component